# OTP 경로탐색용 OD 데이터 생성
> TCN(Trip Chain Network) → OTP(OpenTripPlanner) 입력용 OD pair 집계

**목적**: 스마트카드 통행 데이터를 OTP 경로탐색 요청에 사용할 수 있는 OD pair 형태로 변환

**출력 컬럼**:
- `od_pair`: OD 키 (승차정류장ID_하차정류장ID)
- `o_stop_id`, `d_stop_id`: 정류장 ID
- `o_lat`, `o_lon`, `d_lat`, `d_lon`: 좌표
- `departure_time`: 대표 출발시간
- `trip_count`: 총 통행 건수 (여러 날짜 합산)

---
## 1. 설정

In [1]:
import pandas as pd

%load_ext autoreload
%autoreload 2

# OTP용 OD pair 그룹화 모듈
from module.tcn_to_otp_od import (
    process_tcn_to_otp_input,
    create_otp_od_data,
    merge_multiple_tcn,
    load_and_merge_tcn_files
)

In [3]:
# TCN 파일 경로 설정
dates = ['20250217', '20250218', '20250219', '20250220', '20250221', '20250222', '20250223']

tcn_paths = [f'../data/tcn/{d}/TCN_{d}_route.parquet' for d in dates]

print(f"TCN 파일 수: {len(tcn_paths)}")

TCN 파일 수: 7


---
## 2. OD pair 집계

In [ ]:
# OTP 입력용 OD pair 생성
od_data = process_tcn_to_otp_input(
    tcn_paths=tcn_paths,
    output_path='../data/otp/input/otp_od_input.csv'  # 저장 경로
)

od_data.head()

Loading ../data/tcn/20250217/TCN_20250217_route.parquet...
  11,981,768 trips
Loading ../data/tcn/20250218/TCN_20250218_route.parquet...
  12,209,302 trips
Loading ../data/tcn/20250219/TCN_20250219_route.parquet...
  12,322,236 trips
Loading ../data/tcn/20250220/TCN_20250220_route.parquet...
  12,402,385 trips
Loading ../data/tcn/20250221/TCN_20250221_route.parquet...
  12,057,793 trips
Loading ../data/tcn/20250222/TCN_20250222_route.parquet...
  9,703,161 trips
Loading ../data/tcn/20250223/TCN_20250223_route.parquet...
  7,071,985 trips

Merging 7 files...
Total OD pairs: 7,957,769
Total trips: 77,748,630
Saved to ../data/otp/input/otp_od_input.csv


,od_pair,o_stop_id,d_stop_id,o_lat,o_lon,d_lat,d_lon,departure_time,trip_count
0,10003_10048,10003,10048,37.35292,126.94574,37.403680,126.94964,20250222093152,1
1,10003_1006,10003,1006,37.35292,126.94574,37.515467,126.90765,20250217100320,4
2,10003_10066,10003,10066,37.35292,126.94574,37.401830,126.95107,20250223152227,1
3,10003_10631,10003,10631,37.35292,126.94574,37.478750,126.89464,20250223133708,1
4,10003_10661,10003,10661,37.35292,126.94574,37.463600,126.89756,20250217074728,13


---
## 3. 필터링 및 저장

In [2]:
od_data = pd.read_csv('../data/otp/input/otp_od_input.csv')

In [9]:
# 최소 통행 건수 필터링 (직접 조정)
MIN_TRIPS = 13

od_filtered = od_data[od_data['trip_count'] >= MIN_TRIPS].copy()
od_filtered = od_filtered.reset_index(drop=True)

print(f"필터링 전: {len(od_data):,}")
print(f"필터링 후 (>= {MIN_TRIPS}건): {len(od_filtered):,}")
print(f"제거된 OD pair: {len(od_data) - len(od_filtered):,}")

od_filtered.head()

필터링 전: 7,957,769
필터링 후 (>= 13건): 774,895
제거된 OD pair: 7,182,874


,od_pair,o_stop_id,d_stop_id,o_lat,o_lon,d_lat,d_lon,departure_time,trip_count
0,10003_10661,10003,10661,37.35292,126.94574,37.463600,126.897560,20250217074728,13
1,10003_10700,10003,10700,37.35292,126.94574,37.452220,126.901570,20250217142818,14
2,10003_1451,10003,1451,37.35292,126.94574,37.443636,127.008010,20250221092229,13
3,10003_1457,10003,1457,37.35292,126.94574,37.389869,126.950687,20250217115249,13
4,10003_8001060,10003,8001060,37.35292,126.94574,37.360000,126.948230,20250217124616,23


In [10]:
# 통계 확인
print("=== OD 통계 ===")
print(f"총 OD pair 수: {len(od_filtered):,}")
print(f"총 통행 건수: {od_filtered['trip_count'].sum():,}")
print(f"\n통행 건수 분포:")
print(od_filtered['trip_count'].describe())

# 필터링된 데이터 저장
od_filtered.to_csv('../data/otp/input/otp_od_input_over13.csv', index=False)
print("\n저장 완료: ../data/otp/input/otp_od_input_over13.csv")

=== OD 통계 ===
총 OD pair 수: 774,895
총 통행 건수: 60,399,151

통행 건수 분포:
count    774895.000000
mean         77.944949
std         238.610176
min          13.000000
25%          18.000000
50%          29.000000
75%          61.000000
max       15231.000000
Name: trip_count, dtype: float64

저장 완료: ../data/otp/input/otp_od_input_over13.csv


---
## 4. GTX OD pair 집계
> GTX 이용 통행(gtx_only, bus+gtx, train+gtx, bus+train+gtx)만 필터링하여 OTP 입력 데이터 생성

In [4]:
# GTX OD pair 생성
od_gtx = process_tcn_to_otp_input(
    tcn_paths=tcn_paths,
    output_path='../data/otp/input/otp_od_input_gtx.csv',
    filter_gtx=True
)

# 통계 출력
print(f"\n=== GTX OD 통계 ===")
print(f"GTX OD pair 수: {len(od_gtx):,}")
print(f"총 GTX 통행 건수: {od_gtx['trip_count'].sum():,}")

od_gtx.head()

Loading ../data/tcn/20250217/TCN_20250217_route.parquet...
  44,378 GTX trips
Loading ../data/tcn/20250218/TCN_20250218_route.parquet...
  45,990 GTX trips
Loading ../data/tcn/20250219/TCN_20250219_route.parquet...
  48,865 GTX trips
Loading ../data/tcn/20250220/TCN_20250220_route.parquet...
  51,282 GTX trips
Loading ../data/tcn/20250221/TCN_20250221_route.parquet...
  50,809 GTX trips
Loading ../data/tcn/20250222/TCN_20250222_route.parquet...
  48,084 GTX trips
Loading ../data/tcn/20250223/TCN_20250223_route.parquet...
  32,838 GTX trips

Merging 7 files...
Total OD pairs: 64,996
Total trips: 322,246
Saved to ../data/otp/input/otp_od_input_gtx.csv

=== GTX OD 통계 ===
GTX OD pair 수: 64,996
총 GTX 통행 건수: 322,246


,od_pair,o_stop_id,d_stop_id,o_lat,o_lon,d_lat,d_lon,departure_time,trip_count
0,10018_4114607,10018,4114607,37.504390,126.946850,37.735400,126.729370,20250223195007,1
1,10018_9001,10018,9001,37.504390,126.946850,37.665252,126.748253,20250218142021,1
2,1001_10807,1001,10807,37.555525,126.972171,37.590610,126.906450,20250219090321,1
3,1001_12593,1001,12593,37.555525,126.972171,37.683500,126.773880,20250221213405,1
4,1001_1270,1001,1270,37.555525,126.972171,37.612111,126.834066,20250217180312,8


In [5]:
od_gtx

,od_pair,o_stop_id,d_stop_id,o_lat,o_lon,d_lat,d_lon,departure_time,trip_count
0,10018_4114607,10018,4114607,37.504390,126.946850,37.735400,126.729370,20250223195007,1
1,10018_9001,10018,9001,37.504390,126.946850,37.665252,126.748253,20250218142021,1
2,1001_10807,1001,10807,37.555525,126.972171,37.590610,126.906450,20250219090321,1
3,1001_12593,1001,12593,37.555525,126.972171,37.683500,126.773880,20250221213405,1
4,1001_1270,1001,1270,37.555525,126.972171,37.612111,126.834066,20250217180312,8
...,...,...,...,...,...,...,...,...,...
64991,9950_4104780,9950,4104780,37.513500,126.941880,37.709780,126.756150,20250217201453,1
64992,9950_9014119,9950,9014119,37.513500,126.941880,37.615250,126.915380,20250222162234,1
64993,9957_9001,9957,9001,37.346290,126.942250,37.665252,126.748253,20250222085927,3
64994,9987_1285,9987,1285,37.350260,126.944670,37.888260,126.747200,20250222132323,1
